# Arsenal-excluded experiments — executed evidence review

The corrected experiments were trained by `python -m experiments.isolated_arsenal` and saved in `results/arsenal_isolated/`. This notebook **loads and verifies those runs; it does not retrain**. Earlier notebooks document the historical Arsenal-inclusive study.

To retrain all corrected arms in a separate directory: `python -m experiments.isolated_arsenal --output work/arsenal_isolated_rerun`.

Source training removes all Arsenal fixtures before reconstructing rolling histories and Elo. Source model selection uses non-Arsenal validation. Arsenal-only training and both adaptations use Arsenal validation. Test is used only after fitting. Original target features retain strictly earlier real match histories, so source and target feature distributions differ.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import torch
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'experiments').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from experiments import train
from experiments.isolated_arsenal import OUTPUT
record = json.loads((OUTPUT / 'manifest.json').read_text())
for path, expected in record['code_sha256'].items():
    assert train.digest(ROOT / path) == expected, path
for path, expected in record['input_sha256'].items():
    assert train.digest(train.DATA / path) == expected, path
for path, expected in record['artifact_sha256'].items():
    assert train.digest(OUTPUT / path) == expected, path
print('All recorded code, input and output hashes verified.')
display(pd.Series({k:record[k] for k in ['started_utc','models_frozen_utc','finished_utc']}))
display(pd.DataFrame(record['rows']))

All recorded code, input and output hashes verified.


started_utc          2026-10-26T10:40:17.612400+00:00
models_frozen_utc    2026-10-26T10:40:32.474487+00:00
finished_utc         2026-10-26T10:40:33.147700+00:00
dtype: object

,league,arsenal
train,7142,815
validation,654,74
test,656,75


## Separate datasets

League rows exclude Arsenal as either home or away team. The original 7,977 training rows become 7,142 eligible source rows after exclusion and feature reconstruction. Arsenal retains 815/74/75 fixtures.

In [2]:
identities = []
for population in ['league', 'arsenal']:
    for split in ['train','validation','test']:
        meta = pd.read_csv(OUTPUT / population / split / 'index.csv')
        mask = train.arsenal(meta)
        assert mask.all() if population == 'arsenal' else not mask.any()
        identities.append(set(meta.match_id))
for i, ids in enumerate(identities):
    assert all(not ids.intersection(other) for other in identities[:i])
print('All six partitions have unique, disjoint fixture sets and correct team membership.')

All six partitions have unique, disjoint fixture sets and correct team membership.


## Training executions and every epoch

All MLPs use AdamW, lr 0.0001, weight decay 0.01, batch 256, seed 42, at most 100 epochs and patience 20. Source pretraining is independently executed twice under identical deterministic settings; matching weights are expected. Fine-tuning resets AdamW and retains the source scaler. Validation losses in the training summary refer to the indicated validation population.

In [3]:
summary = pd.read_csv(OUTPUT / 'models/training_summary.csv')
display(summary)
audits = [json.loads((OUTPUT / 'models' / f'{name}_execution.json').read_text()) for name in summary.model]
display(pd.DataFrame(audits)[['model','started_utc','finished_utc','epochs_run','best_epoch','optimizer_steps','training_rows','validation_rows']])

,model,best_epoch,epochs_run,validation_log_loss,training_rows,checkpoint_sha256,training_population,validation_population
0,arsenal_mlp,100,100,0.864435,815,66a872e6c2b7e900069cf1e1c178b02f47494040c1f9a1...,arsenal,arsenal
1,league_mlp,99,100,0.975268,7142,103ad2f65ba73a201cc5dfe0f562310f18120450522d9d...,league,league
2,full_finetune_mlp,2,21,0.796803,815,c2a91cef3b525bb3ed0be94ba3250ad4f80c34da20500b...,arsenal,arsenal
3,league_frozen_source_mlp,99,100,0.975268,7142,f88358c991706acc5a6c81e51d9b9d7b3322256583a214...,league,league
4,head_finetune_mlp,1,20,0.797128,815,18cb0477189d6c4d6f1138509a5fdd52dce13b3db0a65e...,arsenal,arsenal


,model,started_utc,finished_utc,epochs_run,best_epoch,optimizer_steps,training_rows,validation_rows
0,arsenal_mlp,2026-10-26T10:40:17.625301+00:00,2026-10-26T10:40:18.882460+00:00,100,100,400,815,74
1,league_mlp,2026-10-26T10:40:18.886613+00:00,2026-10-26T10:40:21.172940+00:00,100,99,2800,7142,654
2,full_finetune_mlp,2026-10-26T10:40:21.174953+00:00,2026-10-26T10:40:21.263347+00:00,21,2,84,815,74
3,league_frozen_source_mlp,2026-10-26T10:40:21.266057+00:00,2026-10-26T10:40:23.346908+00:00,100,99,2800,7142,654
4,head_finetune_mlp,2026-10-26T10:40:23.348678+00:00,2026-10-26T10:40:23.407805+00:00,20,1,80,815,74


In [4]:
histories = []
for name in summary.model:
    h = pd.read_csv(OUTPUT / 'models' / f'{name}_history.csv')
    h.insert(0, 'model', name)
    histories.append(h)
with pd.option_context('display.max_rows', None):
    display(pd.concat(histories, ignore_index=True))

,model,epoch,training_log_loss,validation_log_loss,validation_accuracy,best_validation_loss,best_epoch_so_far,checkpoint_saved,patience_used,optimizer_steps,epoch_seconds
0,arsenal_mlp,1,1.089138,1.081805,0.486486,1.081805,1,True,0,4,0.009638
1,arsenal_mlp,2,1.086166,1.079540,0.513514,1.079540,2,True,0,8,0.004442
2,arsenal_mlp,3,1.084127,1.077312,0.527027,1.077312,3,True,0,12,0.004244
3,arsenal_mlp,4,1.082826,1.075056,0.527027,1.075056,4,True,0,16,0.004061
4,arsenal_mlp,5,1.080569,1.072784,0.540541,1.072784,5,True,0,20,0.004112
5,arsenal_mlp,6,1.079655,1.070475,0.540541,1.070475,6,True,0,24,0.004179
6,arsenal_mlp,7,1.076868,1.068149,0.540541,1.068149,7,True,0,28,0.004170
7,arsenal_mlp,8,1.074225,1.065802,0.540541,1.065802,8,True,0,32,0.004129
8,arsenal_mlp,9,1.072349,1.063411,0.540541,1.063411,9,True,0,36,0.004146
9,arsenal_mlp,10,1.073181,1.060987,0.554054,1.060987,10,True,0,40,0.004220


## Verify frozen layers and independent source fits

The 39→32 and 32→16 linear layers have `requires_grad=False` during head-only adaptation; only the 16→3 layer enters the optimizer. Training asserts that frozen gradients are absent and hidden weights remain unchanged after every epoch. The checks below independently compare the saved checkpoints.

In [5]:
def checkpoint(name):
    return torch.load(OUTPUT / 'models' / f'{name}_best.pt', map_location='cpu', weights_only=False)
source = checkpoint('league_frozen_source_mlp')
head = checkpoint('head_finetune_mlp')
checks = {key: torch.equal(source['model_state'][key], head['model_state'][key]) for key in ['0.weight','0.bias','3.weight','3.bias']}
assert all(checks.values())
assert any(not torch.equal(source['model_state'][k], head['model_state'][k]) for k in ['5.weight','5.bias'])
audit = next(a for a in audits if a['model']=='head_finetune_mlp')
assert audit['trainable_parameters'] == ['5.weight','5.bias']
assert all(torch.equal(value, checkpoint('league_mlp')['model_state'][key]) for key,value in source['model_state'].items())
display(pd.Series(checks, name='hidden_parameter_unchanged'))
print('Output parameters changed; independently trained source weights match.')
print('Trainable parameters:', audit['trainable_parameters'])

0.weight    True
0.bias      True
3.weight    True
3.bias      True
Name: hidden_parameter_unchanged, dtype: bool

Output parameters changed; independently trained source weights match.
Trainable parameters: ['5.weight', '5.bias']


## Arsenal evaluation: independently reload models and recompute scores

Random Forest and dummy are fitted only on source training and applied directly to Arsenal. Dummy probabilities are the source A/D/H class frequencies. Saved prediction rows are checked against the original Arsenal validation and test fixtures.

In [6]:
import joblib
metrics = pd.read_csv(OUTPUT / 'evaluation/metrics.csv')
schema, splits = train.load()
for row in metrics.itertuples():
    x,y,meta = splits[row.split]
    mask = train.arsenal(meta)
    frame = pd.DataFrame(x[mask], columns=schema['features'])
    if row.model.endswith('mlp'):
        p = train.predict_checkpoint(OUTPUT / 'models' / f'{row.model}_best.pt', frame).to_numpy()
    else:
        p = joblib.load(OUTPUT / 'models' / f'{row.model}.joblib').predict_proba(x[mask])
    saved = pd.read_csv(OUTPUT / 'evaluation' / f'{row.model}_{row.split}_predictions.csv')
    assert saved.match_id.tolist() == meta.loc[mask,'match_id'].tolist()
    np.testing.assert_allclose(saved[['p_A','p_D','p_H']].to_numpy().astype(p.dtype),p,rtol=0,atol=1e-15)
    scores = train.score(y[mask], p)
    for metric in ['log_loss','brier_multiclass','accuracy','n']:
        np.testing.assert_allclose(scores[metric],getattr(row,metric),rtol=0,atol=1e-12)
print('Every saved model reproduces its fixture probabilities and reported scores.')
display(metrics)

Every saved model reproduces its fixture probabilities and reported scores.


,model,split,population,log_loss,brier_multiclass,accuracy,n
0,arsenal_mlp,validation,Arsenal,0.864435,0.503668,0.702703,74
1,league_mlp,validation,Arsenal,0.797180,0.453334,0.675676,74
2,full_finetune_mlp,validation,Arsenal,0.796803,0.453112,0.689189,74
3,league_frozen_source_mlp,validation,Arsenal,0.797180,0.453334,0.675676,74
4,head_finetune_mlp,validation,Arsenal,0.797128,0.453307,0.675676,74
5,league_random_forest,validation,Arsenal,0.828401,0.471868,0.702703,74
6,league_dummy,validation,Arsenal,1.040882,0.628818,0.472973,74
7,arsenal_mlp,test,Arsenal,0.959723,0.569044,0.600000,75
8,league_mlp,test,Arsenal,0.908240,0.533728,0.613333,75
9,full_finetune_mlp,test,Arsenal,0.908132,0.533687,0.600000,75


This is a single-seed exploratory experiment on a previously inspected test period. Full and head-only fine-tuning have very small effects in this run. Raw numerical evidence is shown above; this notebook does not change the saved models or select models using test scores.